In [1]:
from langchain.llms import OpenAI
from langchain.prompts import PromptTemplate
from langchain_community.chat_models import ChatOpenAI
from langchain.chains import LLMChain, SequentialChain
import os
import json
from flask import Flask, request, jsonify, render_template, redirect, url_for, session
from werkzeug.utils import secure_filename
from auth import validar_usuario
from dotenv import load_dotenv

# Adicionando memória ao Chat

In [2]:
from langchain.memory import ConversationBufferMemory
memory = ConversationBufferMemory(
    return_messages=True,
    memory_key='chat_history'
)

/var/folders/y9/1fdr15b509d47wr6w2nglnmc0000gn/T/ipykernel_85103/3110294988.py:2: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(


# Acessando a API OpenAi

In [3]:
# Carregar variáveis de ambiente do arquivo .env
load_dotenv()

# Configura a chave de API do OpenAI
# (Garante que a variável de ambiente OPENAI_API_KEY esteja definida)
openai_api_key = os.getenv("OPENAI_API_KEY")

# Cria o objeto LLM com o modelo desejado
llm = OpenAI(model_name="gpt-3.5-turbo", temperature=0.7, openai_api_key=openai_api_key)

app = Flask(__name__)
app.secret_key = "chave.env"

/Users/daniloportela/Desktop/TESTES_IA/meu_app/.venv/lib/python3.13/site-packages/langchain_community/llms/openai.py:255: UserWarning: You are trying to use a chat model. This way of initializing it is no longer supported. Instead, please use: `from langchain_community.chat_models import ChatOpenAI`
  warnings.warn(
/Users/daniloportela/Desktop/TESTES_IA/meu_app/.venv/lib/python3.13/site-packages/langchain_community/llms/openai.py:1089: UserWarning: You are trying to use a chat model. This way of initializing it is no longer supported. Instead, please use: `from langchain_community.chat_models import ChatOpenAI`
  warnings.warn(


# Chamador de Ferramentas
Com acesso ao Python

In [4]:
from langchain_experimental.tools import PythonAstREPLTool

tools = [PythonAstREPLTool()]

ModuleNotFoundError: No module named 'langchain_experimental'

# Criando os Agentes

In [5]:
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

diretor = '''Você é um Diretro de Criatividade, responsável por coordenar e integrar as
diferentes áreas de criação (Pesquisa, Planejamento, Redator, Design e Apresentação).
Sua missão é garantir que todos os aspectos da campanha estejam alinhados e
harmonizados, desde a pesquisa até a execução final.

Você deve: Gerenciar o fluxo de trabalho, coordenar as equipes, garantir a
integração entre as áreas, revisar e ajustar as entregas de cada área,
identificar e resolver problemas, manter a comunicação clara e fluida entre as
equipes, garantir que o resultado final esteja alinhado com a visão criativa
e os objetivos da campanha.

Você deve ser capaz de:
- Compreender o briefing e os objetivos da campanha
- Identificar as necessidades de cada área
- Coordenar o trabalho entre as equipes
- Revisar e ajustar as entregas de cada área
- Garantir a qualidade e a coerência do resultado final
- Identificar e resolver problemas
- Manter a comunicação clara e fluida entre as equipes
- Garantir que o resultado final esteja alinhado com a visão criativa e os objetivos da campanha
- Ser flexível e adaptável a mudanças

Diretrizes:
- Entenda o briefing, o objetivo, a plataforma e o público
- Garanta que cada etapa do fluxo de trabalho esteja clara e bem definida.

VERIFIQUE:
1. O input recebido está completo?
2. Está alinhado ao briefing inicial?
3. Precisa de ajustes?

Se ok, direcione ao próximo agente com instruções claras.
Se ajustar, explique o problema e solicite revisão.

Exemplo de respostas:
- "ok: Por favor, desenvolva um post para Instagram usando estes dados [detalhes]"
- "ajustar (PESQUISA): Faltam dados sobre o público-alvo. Complete a pesquisa primeiro.
- "ajustar (REDATOR): O texto não está claro. Reescreva com foco no público jovem."'''

redator = '''Você é um Redator Publicitário experiente, criativo e genial. 
Sua missão é criar textos criativos e persuasivos que comuniquem com clareza, 
influenciem o público-alvo e estejam alinhados à estratégia da campanha.

Você é capaz de:
- Como adaptar estilo e tom de voz a diferentes públicos e marcas
- A importância do storytelling para engajar
- O papel da redação nas estratégias de marketing digital
- Como aplicar técnicas de SEO e gatilhos mentais
- A função de cada formato (post, roteiro, institucional etc.)

Diretrizes:
- Entenda o briefing, o objetivo, a plataforma e o público
- Use linguagem simples, criativa e acessível
- Crie títulos fortes e textos objetivos
- Enriqueça com listas, insights ou exemplos, se couber
- Seja divertido e original
- Sempre indique uma sugestão de imagem
# Formatos esperados:
- Redes Sociais: título, legenda, hashtags, sugestão de imagem
- Roteiro: cena, texto, sugestão de imagem

Restrições:
- Escreva em pt-br
- Seja direto e evite rodeios
- Se não souber algo, diga com clareza

FORMATO DE SAÍDA:
[VERSÃO 1.0 - REDAÇÃO]
Título: ...
Corpo: ...
Hashtags: ...

OBSERVAÇÕES:
- Numere suas versões (ex: 1.0, 1.1)
- Inclua "[REVISÃO SOLICITADA]" no final
- Responda como se estivesse escrevendo para um cliente real, de maneira estratégica e criativa.
Certifique-se de usar a ferramenta PythonAstREPLTool para auxiliar a responder as perguntas
- Se houver necessidade de revisão, faça sugestões claras e objetivas'''

pesquisa = '''Você é um especialista em pesquisa de mercado altamente qualificado, com foco em comportamento
do consumidor, plataformas digitais e movimentos culturais. 
Sua missão é analisar o briefing, coletar informações relevantes sobre o mercado, concorrência e tendências 
para ajudar na criação de uma estratégia de marketing criativo.

ANTES DE ENTREGAR: 
- Seu relatório deve incluir:
  1. Fontes verificadas
  2. Dados quantitativos (se aplicável)
  3. Principais insights
- Finalize com: "[AGUARDANDO APROVAÇÃO DO DIRETOR]"'''

designer = '''Você é um Designer criativo, ousado e com profundo entendimento de estética, 
composição visual e identidade de marca.
Sua missão é interpretar o briefing da campanha e propor **conceitos visuais** impactantes e 
alinhados à identidade visual do cliente. 
Você também deve sugerir **prompts detalhados** que podem ser usados para gerar imagens em 
ferramentas como DALL·E ou Midjourney.
Se houver dados de identidade visual do cliente, considere:
- Paleta de cores (ex: tons escuros, pastéis, vibrantes)
- Tipografia preferida (ex: moderna, manuscrita, institucional)
- Estilo gráfico (ex: minimalista, retrô, urbano)
- Formatos comuns (ex: 1:1, 9:16, colagem, banner horizontal)
Sua resposta deve incluir:
1. Conceito visual da campanha (mood geral)
2. Sugestão de cores, estilo e elementos gráficos
3. Ideias de layout (ex: destaque no produto, sobreposição de texto)
4. 2 ou 3 sugestões de imagem com prompt para Midjourney/DALL·E
5. Sugestões de formatos (feed, stories, reels, etc.)
Observacões:
- Fale sempre em português
Se não houver identidade visual definida, use bom senso e proponha um estilo coerente com o briefing.
'''

apresentacao = '''Você é um Especialista em Apresentações, responsável por transformar todo o 
conteúdo em uma apresentação clara, profissional e impactante.

EXECUTAR APENAS QUANDO:
- Todos os módulos anteriores estiverem marcados como "APROVADO PELO DIRETOR"
- Receber a instrução explícita: "[ok] Apresentação"

FORMATO:
[SLIDE 1] Título: ...
[SLIDE 2] Dados: ...

Instruções de formato: 
Siga sempre a estrutura: [APRESENTACAO] Slide 1: ...Slide 2: ...etc.
Você deve criar uma apresentação de slides com base no conteúdo fornecido.

ENTREGA PADRÃO:
[PROPOSTA VISUAL A]
- Cores: ...
- Estilo: ...

[PROPOSTA VISUAL B]
- Cores: ...
- Estilo: ...

INSTRUÇÃO FINAL:
"Escolha entre A ou B ou solicite ajustes.'''


# Fluxo de trabalho com Aprovações

In [ ]:
sequenceDiagram
    Usuário->>Diretor: Briefing inicial
    Diretor->>Pesquisa: "Pesquise sobre X"
    Pesquisa-->>Diretor: Dados brutos
    Diretor->>Redator: "Crie texto baseado nestes dados"
    Redator-->>Diretor: Texto proposto
    Diretor->>Designer: "Desenvolva conceito visual para este texto"
    Designer-->>Diretor: Proposta visual
    Diretor->>Apresentação: "Formate tudo em slides"
    Apresentação-->>Usuário: Apresentação final

# Fluxo de direcionamento e aprovação

In [ ]:
from streamlit import chat_message


def check_approval(last_output):
    if "AGUARDANDO APROVAÇÃO" in last_output:
        return "diretor"
    elif "REVISÃO SOLICITADA" in last_output:
        return "diretor"
    else:
        return None  # Continua o fluxo

# Dentro do loop de execução:
next_step = check_approval(chat_message)
if next_step == "diretor":
    send_to_diretor_for_review()


prompt = ChatPromptTemplate.from_messages([
    ('system', redator),
    ('placeholder', '{chat_history}'),
    ('human', '{input}'),
    ('placeholder', '{agent_scratchpad}')
])

chat = ChatOpenAI()
agent = create_tool_calling_agent(chat, tools, prompt)
agente_redator = AgentExecutor(agent=agent, tools=tools, verbose=True)



# Código de Votação

In [ ]:
from collections import Counter
import time

def realizar_votacao(diretor, pergunta, opcoes, agentes):
    # 1. Diretor inicia a votação
    print(f"\n[VOTAÇÃO INICIADA] {pergunta}")
    for idx, opcao in enumerate(opcoes, 1):
        print(f"Opção {idx}: {opcao[:100]}...")  # Mostra preview
    
    votos = []
    
    # 2. Coleta votos dos agentes
    for agente_nome, agente_instancia in agentes.items():
        try:
            resposta = agente_instancia.invoke({
                "input": f"[escolha] {pergunta}\nOpções:\n1. {opcoes[0]}\n2. {opcoes[1]}"
            })
            
            if "[MEU VOTO]" in resposta.content:
                voto = resposta.content.split("[MEU VOTO] ")[1].split("\n")[0].strip()
                justificativa = resposta.content.split("[JUSTIFICATIVA] ")[1].strip()
                votos.append(voto)
                print(f"{agente_nome} votou em: {voto} | Razão: {justificativa}")
            else:
                print(f"{agente_nome} se absteve")
                
        except Exception as e:
            print(f"Erro no voto de {agente_nome}: {str(e)}")
    
    # 3. Apuração
    contagem = Counter(votos)
    resultado = contagem.most_common(1)[0][0] if votos else None
    
    # 4. Resolução de empates (Diretor decide)
    if len(contagem) > 1 and contagem.most_common()[0][1] == contagem.most_common()[1][1]:
        decisao = maestro.invoke({
            "input": f"DESEMPATE: Escolha entre {contagem.most_common()[0][0]} e {contagem.most_common()[1][0]}"
        })
        resultado = decisao.content
        print(f"EMPATE! Diretor decidiu por: {resultado}")
    
    return resultado

# Setup dos Agentes

In [ ]:
agente_redator.invoke({'input': 'Crie um texto criativo e persuasivo para uma campanha de marketing de um novo produto,que é uma guitarra Fender. O público-alvo são jovens adultos entre 18 e 30 anos.'})   

In [ ]:
agentes = {
    "Pesquisa": AgentExecutor(pesquisa, tools=[]),
    "Redator": AgentExecutor(redator, tools=[]),
    "Designer": AgentExecutor(designer, tools=[]),
    "Apresentação": AgentExecutor(apresentacao, tools=[])
}

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
import chat

prompt = ChatPromptTemplate.from_messages([
    ('system', 'Você é um Redator Publicitário experiente, criativo e genial. Sua habilidade é escrever textos criativos e persuasivos com clareza, gramática correta e estilo adaptável a diferentes públicos e marcas para diferentes formatos. Criar mensagens que motivem a ação do público-alvo, com conhecimento sobre como influenciar e persuadir um público, através de parcerias com outras marcas ou influenciadores e capacidade de contar histórias envolventes.'),
    ('user', '{input}'),
    ])

chain = prompt | chat.bind(model)
def get_response(prompt):
    response = chain.invoke({"input": prompt})
    return response
def main():
    prompt = "Crie um texto criativo e persuasivo para uma campanha publicitária de uma nova linha de produtos de beleza, destacando os benefícios e diferenciais da marca."
    response = get_response(prompt)
    print(f"Prompt: {prompt}")
    print(f"Response: {response}")
if __name__ == "__main__":
    main()


# Definição dos PromptTemplates para cada etapa do fluxo

In [ ]:
prompt_tendencias = PromptTemplate(
    input_variables=["briefing"],
    template=(
        "Dado o seguinte briefing: {briefing}, liste as principais tendências de mercado "
        "relevantes para uma campanha publicitária atual."
    )
)

prompt_planning = PromptTemplate(
    input_variables=["briefing", "tendencias"],
    template=(
        "Com base no briefing: {briefing} e nas tendências: {tendencias}, elabore um plano estratégico "
        "para uma campanha publicitária, definindo objetivos, público-alvo, canais e cronograma."
    )
)

prompt_conteudo = PromptTemplate(
    input_variables=["plano"],
    template=(
        "Com base no seguinte plano estratégico: {plano}, gere conteúdos publicitários criativos, "
        "incluindo sugestões de textos e imagens para a campanha."
    )
)

# Criação dos LLMChain para cada etapa

In [ ]:
chain_tendencias = LLMChain(llm=llm, prompt=prompt_tendencias)
chain_planning = LLMChain(llm=llm, prompt=prompt_planning)
chain_conteudo = LLMChain(llm=llm, prompt=prompt_conteudo)

class GerenciadorFluxo:
    """
    Classe para gerenciar o fluxo iterativo do processo criativo.
    Etapas:
      - briefing: Recebe e armazena o briefing inicial.
      - tendencias: Gera insights a partir do briefing.
      - planejamento: Cria o plano estratégico.
      - conteudo: Gera textos e sugestões visuais.
      - finalizado: Processo concluído.
    """

    def __init__(self):
        self.briefing = None
        self.tendencias = None
        self.plano = None
        self.conteudos = None
        self.etapa = "briefing"  # etapas: briefing, planejamento, conteudo, finalizado

    def processar_mensagem(self, mensagem):
        if self.etapa == "briefing":
            self.briefing = mensagem
            # Chama o chain de tendências para analisar o briefing
            tendencias_result = chain_tendencias.run(briefing=self.briefing)
            self.tendencias = tendencias_result.strip()
            self.etapa = "planejamento"
            return (
                f"Briefing registrado: '{self.briefing}'.\n"
                f"As tendências identificadas são: {self.tendencias}\n"
                "Deseja prosseguir para a criação do plano estratégico? (Envie 'sim' para confirmar.)"
            )

        elif self.etapa == "planejamento":
            if mensagem.strip().lower() == "sim":
                planning_result = chain_planning.run(
                    briefing=self.briefing, tendencias=self.tendencias
                )
                self.plano = planning_result.strip()
                self.etapa = "conteudo"
                return (
                    f"Plano estratégico criado: {self.plano}\n"
                    "Deseja prosseguir para a geração dos conteúdos (textos e imagens)? (Envie 'sim' para confirmar.)"
                )
            else:
                return (
                    "Processo em planejamento: se deseja ajustar o plano, informe quais alterações deseja aplicar."
                )

        elif self.etapa == "conteudo":
            if mensagem.strip().lower() == "sim":
                conteudo_result = chain_conteudo.run(plano=self.plano)
                self.conteudos = conteudo_result.strip()
                self.etapa = "finalizado"
                return (
                    f"Conteúdos gerados: {self.conteudos}\n"
                    "Caso deseje ajustar alguma etapa, informe qual (briefing, planejamento ou conteúdo)."
                )
            else:
                return (
                    "Processo na etapa de conteúdos: se deseja ajustes, informe quais alterações deseja aplicar."
                )

        elif self.etapa == "finalizado":
            return (
                "Processo finalizado. Para reiniciar ou modificar alguma etapa, por favor, informe o que deseja fazer."
            )

        else:
            return "Etapa desconhecida. Reinicie o processo, por favor."

# Fluxo do gerenciador.
Em produção, recomenda-se gerenciar o estado por sessão para cada usuário.

In [ ]:
chain_tendencias = LLMChain(llm=llm, prompt=prompt_tendencias)
chain_planning = LLMChain(llm=llm, prompt=prompt_planning)
chain_conteudo = LLMChain(llm=llm, prompt=prompt_conteudo)


class GerenciadorFluxo:
    """
    Classe para gerenciar o fluxo iterativo do processo criativo.
    Etapas:
      - briefing: Recebe e armazena o briefing inicial.
      - planejamento: Cria o plano estratégico com base nas tendências.
      - conteudo: Gera textos e sugestões visuais.
      - finalizado: Processo concluído.
    """

    def __init__(self):
        self.briefing = None
        self.tendencias = None
        self.plano = None
        self.conteudos = None
        self.etapa = "briefing"  # etapas: briefing, planejamento, conteudo, finalizado

    def processar_mensagem(self, mensagem):
        if self.etapa == "briefing":
            self.briefing = mensagem
            # Chama o chain de tendências para analisar o briefing
            tendencias_result = chain_tendencias.run(briefing=self.briefing)
            self.tendencias = tendencias_result.strip()
            self.etapa = "planejamento"
            return (
                f"Briefing registrado: '{self.briefing}'.\n"
                f"As tendências identificadas são: {self.tendencias}\n"
                "Deseja prosseguir para a criação do plano estratégico? (Envie 'sim' para confirmar.)"
            )

        elif self.etapa == "planejamento":
            if mensagem.strip().lower() == "sim":
                planning_result = chain_planning.run(
                    briefing=self.briefing, tendencias=self.tendencias
                )
                self.plano = planning_result.strip()
                self.etapa = "conteudo"
                return (
                    f"Plano estratégico criado: {self.plano}\n"
                    "Deseja prosseguir para a geração dos conteúdos (textos e imagens)? (Envie 'sim' para confirmar.)"
                )
            else:
                return (
                    "Processo em planejamento: se deseja ajustar o plano, informe quais alterações deseja aplicar."
                )

        elif self.etapa == "conteudo":
            if mensagem.strip().lower() == "sim":
                conteudo_result = chain_conteudo.run(plano=self.plano)
                self.conteudos = conteudo_result.strip()
                self.etapa = "finalizado"
                return (
                    f"Conteúdos gerados: {self.conteudos}\n"
                    "Caso deseje ajustar alguma etapa, informe qual (briefing, planejamento ou conteúdo)."
                )
            else:
                return (
                    "Processo na etapa de conteúdos: se deseja ajustes, informe quais alterações deseja aplicar."
                )

        elif self.etapa == "finalizado":
            return (
                "Processo finalizado. Para reiniciar ou modificar alguma etapa, por favor, informe o que deseja fazer."
            )

        else:
            return "Etapa desconhecida. Reinicie o processo, por favor."

In [ ]:
fluxo = GerenciadorFluxo()

@app.route("/")
def chat_page():
    # Renderiza a página HTML que já está na pasta templates
    return render_template("chat.html")


@app.route("/api/chat", methods=["POST"])
def process_chat():
    dados = request.get_json()
    # Utilize 'mensagem' conforme enviado pelo chat.js; 'cliente_id' pode ser tratado se necessário
    mensagem = dados.get("mensagem", "")
    # Aqui, o campo 'cliente_id' está disponível, mas para este exemplo não estamos utilizando-o
    resposta = fluxo.processar_mensagem(mensagem)
    # Retorne a resposta com a chave 'resposta' conforme esperado pelo chat.js
    return jsonify({"resposta": resposta})


if __name__ == "__main__":
    app.run(debug=True)